# TissueSpectFTry the pipeline online: check a change, inspect the labels, run the baselines.Top to bottom in ~15 min. Sections 1–2 need no data.> Colab has 2 cores, so `maxt` is skipped and the spectral stages run on a few> chromosomes. **A partial run is a rehearsal, not a result.**

## 1 · Setup

In [ ]:
import shutil, subprocessif shutil.which("Rscript") is None:    print("Installing R (~2 min)...")    subprocess.run("apt-get -qq update && apt-get -qq install -y r-base-core", shell=True)print(subprocess.run(["Rscript","--version"], capture_output=True, text=True).stderr.strip())

In [ ]:
!git clone --depth 1 --quiet https://github.com/Danpc11/TissueSpectF.git 2>/dev/null || echo "already cloned"%cd TissueSpectF!chmod +x tsf && ./tsf --help | head -25

## 2 · Does it work?No data needed. Run this after any change.

In [ ]:
!make test

In [ ]:
!pip -q install pytest 2>/dev/null!python3 -m pytest tests/ml/ -q

**Self-check** — plants known components in synthetic cohorts and asserts thepipeline recovers them. ~10 min.

In [ ]:
!TSF_MAXT_B=100 TSF_CONDITION_B=300 ./tsf selfcheck 2>&1 | tail -25

## 3 · Get the data

In [ ]:
import osos.environ["TSF_GEO_DIR"]     = "/content/data"os.environ["TSF_INTERIM_DIR"] = "/content/interim"os.environ["TSF_RESULTS_DIR"] = "/content/results"# Keep results after the session:# from google.colab import drive; drive.mount('/content/drive')# os.environ["TSF_RESULTS_DIR"] = "/content/drive/MyDrive/TissueSpectF/results"

In [ ]:
!./tsf fetch --geo-dir /content/data!./tsf check --geo-dir /content/data

**Read a series before trusting a config.** Change the accession and the twofields to inspect any other.

In [ ]:
!Rscript scripts/inspect_series_matrix.R \    /content/data/GSE162694_series_matrix.txt.gz "fibrosis" "nas score"

## 4 · LabelsThe output to read carefully. Compare with the cohort table in the README —a count that differs means a label went somewhere unexpected.

In [ ]:
!./tsf ingest 2>&1 | grep -E "Ingesting|Labels|: [0-9]+$|Grid coverage|filter|keep_conditions"

In [ ]:
import pandas as pd, glob, osrows = [pd.read_csv(f, sep="\t").query("keep").groupby("class_id").size()          .rename(os.path.basename(os.path.dirname(f)))        for f in sorted(glob.glob("/content/interim/*/samples.tsv"))]t = pd.concat(rows, axis=1).fillna(0).astype(int)t["total"] = t.sum(axis=1)t

## 5 · Spectra`--chromosomes` is what makes this finish here.

In [ ]:
!./tsf run --from spectra --to spectra --chromosomes 1,17 2>&1 | tail -15

Power against period, for one condition and chromosome.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob, osCHR = "17"   # pandas reads numeric chromosomes as ints, so compare as stringspaths = sorted(glob.glob("/content/results/*/spectra/spectra_condition_*.tsv"))d = pd.read_csv(paths[0], sep="\t")d = d[(d["chr"].astype(str) == CHR) & (d["sample"] == "avg_signal")]fig, ax = plt.subplots(figsize=(11, 3))ax.plot(d.period, d.power, lw=0.8)ax.set(xscale="log", xlabel="period (genes per cycle)", ylabel="power",       title=f"{os.path.basename(paths[0])}  ·  chr{CHR}")ax.invert_xaxis(); plt.tight_layout(); plt.show()

**Spectral window** — what the gaps alone can produce. Run it before readinganything into a peak.

In [ ]:
!./tsf window --chromosomes 1,17 2>&1 | tail -12

## 6 · ConsensusPower, prevalence and phase-locking per frequency, from the per-sample spectra.Two warnings are expected: they say a claim is *not reachable*, which is not thesame as absent.

In [ ]:
!./tsf consensus --chromosomes 1,17 --n-null 199 2>&1 | tail -20

## 7 · BaselinesWorth running here even when nothing else is. No torch, no model.Read **`lift`** over the majority class, not accuracy — and the per-class reportbelow it.

In [ ]:
!./tsf ae-prepare!python3 scripts/run_baselines.py --data /content/results/autoencoder/data \                                  --out  /content/results/autoencoder/baselines

## 8 · Try something```bash# the other gene universe./tsf run --from ingest --to spectra --gene-universe '^(protein-coding|ncRNA)$' \    --interim-dir /content/interim_nc --results-dir /content/results_nc --chromosomes 1,17# another stability criterion, nothing upstream recomputed./tsf run --from stability --criterion consistency --stable-frac 0.7 --chromosomes 1,17# one cohort on its own./tsf ingest GSE130970```Every path and parameter is a flag: `./tsf --help`.

---### What this notebook cannot tell you| question | needs ||---|---|| Is a component real? | full genome, `maxt`, the permutation counts in `RUNBOOK.md` || Does the classifier work? | leave-one-cohort-out on the whole grid || Is a peak biological? | `./tsf window`, then replication across cohorts |What it *does* tell you: the code runs, the labels are what you think they are,and the machinery recovers a signal it is known to contain.